In [ ]:
# List all files under data/rawdomains/* 

from pathlib import Path
import os
import pandas as pd

# Locate repo root 
CWD = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find repo root (no parent directory contains a /data folder).")

REPO_ROOT = find_repo_root(CWD)
RAWDOMAINS_DIR = REPO_ROOT / "data" / "rawdomains"

print("Notebook CWD:", CWD)
print("Repo root:", REPO_ROOT)
print("Rawdomains dir:", RAWDOMAINS_DIR)

if not RAWDOMAINS_DIR.exists():
    raise FileNotFoundError(f"Missing folder: {RAWDOMAINS_DIR}")

# Collect files
rows = []
for path in RAWDOMAINS_DIR.rglob("*"):
    if path.is_file():
        rel = path.relative_to(REPO_ROOT)
        rows.append({
            "relative_path": str(rel),
            "domain_folder": path.relative_to(RAWDOMAINS_DIR).parts[0] if len(path.relative_to(RAWDOMAINS_DIR).parts) else "",
            "filename": path.name,
            "ext": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 4),
        })

files_df = pd.DataFrame(rows).sort_values(["domain_folder", "relative_path"]).reset_index(drop=True)

def print_tree(base: Path):
    print("\n data/rawdomains tree")
    for root, dirs, files in os.walk(base):
        rel_root = Path(root).relative_to(base)
        indent = "  " * len(rel_root.parts)
        folder_name = "." if str(rel_root) == "." else rel_root.name
        print(f"{indent}{folder_name}/")
        for f in sorted(files):
            fp = Path(root) / f
            mb = fp.stat().st_size / (1024 * 1024)
            print(f"{indent}  - {f} ({mb:.3f} MB)")

print_tree(RAWDOMAINS_DIR)

# Summary by Domain
print("\n Counts by domain folder")
if len(files_df) == 0:
    print("No files found under data/rawdomains.")
else:
    display(files_df.groupby(["domain_folder", "ext"]).size().reset_index(name="count").sort_values(["domain_folder","count"], ascending=[True, False]))

# Show full list (filter/sort in notebook UI)
display(files_df)

# Save inventory CSV next to other processed data
out_path = REPO_ROOT / "data" / "processed" / "rawdomains_file_inventory.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
files_df.to_csv(out_path, index=False)
print(f"\nSaved inventory to: {out_path}")

Notebook CWD: /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks
Repo root: /Users/laurenvo/Documents/github/youth_opportunity_index
Rawdomains dir: /Users/laurenvo/Documents/github/youth_opportunity_index/data/rawdomains

 data/rawdomains tree
./
  health/
    - PLACES__Local_Data_for_Better_Health,_Census_Tract_Data,_2025_release_20260311.csv (8.121 MB)
    ACSST5Y2024.S2701_2026-03-11T050431/
      - ACSST5Y2024.S2701-Column-Metadata.csv (0.094 MB)
      - ACSST5Y2024.S2701-Data.csv (2.739 MB)
      - ACSST5Y2024.S2701-Table-Notes.txt (0.011 MB)
    ACSST5Y2024.S1810_2026-03-11T050449/
      - ACSST5Y2024.S1810-Column-Metadata.csv (0.073 MB)
      - ACSST5Y2024.S1810-Data.csv (1.801 MB)
      - ACSST5Y2024.S1810-Table-Notes.txt (0.011 MB)
  economic/
    ACSDT5Y2024.B19013_2026-03-11T045723/
      - ACSDT5Y2024.B19013-Column-Metadata.csv (0.000 MB)
      - ACSDT5Y2024.B19013-Data.csv (0.065 MB)
      - ACSDT5Y2024.B19013-Table-Notes.txt (0.011 MB)
    ACSST5Y2024.S17

,domain_folder,ext,count
0,economic,.csv,8
1,economic,.txt,4
2,education,.csv,6
3,education,.txt,3
4,health,.csv,5
5,health,.txt,2
6,housing,.csv,8
7,housing,.txt,4
9,mobility,.csv,7
14,mobility,.txt,3


,relative_path,domain_folder,filename,ext,size_mb
0,data/rawdomains/economic/ACSDT5Y2024.B19013_20...,economic,ACSDT5Y2024.B19013-Column-Metadata.csv,.csv,0.0003
1,data/rawdomains/economic/ACSDT5Y2024.B19013_20...,economic,ACSDT5Y2024.B19013-Data.csv,.csv,0.0648
2,data/rawdomains/economic/ACSDT5Y2024.B19013_20...,economic,ACSDT5Y2024.B19013-Table-Notes.txt,.txt,0.0110
3,data/rawdomains/economic/ACSST5Y2024.S1701_202...,economic,ACSST5Y2024.S1701-Column-Metadata.csv,.csv,0.0557
4,data/rawdomains/economic/ACSST5Y2024.S1701_202...,economic,ACSST5Y2024.S1701-Data.csv,.csv,1.6295
...,...,...,...,...,...
72,data/rawdomains/safety/calenviroscreen40shpf20...,safety,CES4 Final Shapefile.shp,.shp,8.0562
73,data/rawdomains/safety/calenviroscreen40shpf20...,safety,CES4 Final Shapefile.shp.xml,.xml,0.0698
74,data/rawdomains/safety/calenviroscreen40shpf20...,safety,CES4 Final Shapefile.shx,.shx,0.0614
75,data/rawdomains/youth/services_master.csv,youth,services_master.csv,.csv,4.6610



Saved inventory to: /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/rawdomains_file_inventory.csv
